# Translate on this laptop using a trained Part 1 checkpoint
Run all cells after the final checkpoints have been downloaded. This notebook performs inference only. It chooses the completed model with the lowest full validation NLL unless `MODEL_ID` is set explicitly. Edit `TEXTS` for your own Vietnamese or Japanese inputs. The tokenizer preserves the original scripts, and the model infers the source language from them.

CPU inference is supported. For a quick demonstration, keep inputs and batches short. Unknown or out-of-domain sentences can perform differently from the test sample. The examples below are user-editable demonstrations, not additional held-out evaluation scores.


In [1]:
from pathlib import Path
import json, os
here=Path.cwd().resolve()
ROOT=next((p for p in [here,*here.parents,here/'part 1'] if (p/'shared'/'implementation.ipynb').exists()),None)
assert ROOT is not None, 'Set the notebook working directory inside part 1.'
for cell in json.loads((ROOT/'shared'/'implementation.ipynb').read_text())['cells']:
    if cell['cell_type']=='code' and 'definitions' in cell.get('metadata',{}).get('tags',[]):
        exec(compile(cell['source'],'shared/implementation.ipynb','exec'),globals())


In [2]:
MODEL_ID=None  # Set to 1, 2, 3, 4 or 5 to choose a specific architecture.
TEXTS=['Tôi đang học tiếng Anh.', '私は英語を勉強しています。', 'Cảm ơn bạn rất nhiều.', 'どうもありがとうございます。']
if MODEL_ID is None:
    candidates=[]
    for i in range(1,6):
        path=ROOT/f'model {i}'/f'model_{i}.ipynb'
        if not path.exists(): continue
        for cell in json.loads(path.read_text())['cells']:
            for output in cell.get('outputs',[]):
                record=output.get('data',{}).get('application/json',{})
                if record.get('status')=='full run completed': candidates.append((record['validation']['nll'],i))
    assert candidates,'First run the five experiments and download their checkpoints.'
    MODEL_ID=min(candidates)[1]
path=ROOT/f'model {MODEL_ID}'/'full'/'checkpoint.pt'
assert path.exists(),f'Missing checkpoint: {path}'
saved=torch.load(path,map_location='cpu',weights_only=False)
cfg=Config(**saved['config']); model=Transformer(cfg).to(DEVICE).eval(); model.load_state_dict(saved['model'])
tok=Tokenizer.from_file(str(ROOT/'shared'/'tokenizer_clean.json'))
assert sha(ROOT/'shared'/'tokenizer_clean.json')==saved['tokenizer_sha256']
print('Selected model:',MODEL_ID,'| device:',DEVICE,'| training tokens:',saved['summary']['input_tokens'])
del saved


Selected model: 5 | device: cuda | training tokens: 59998213


In [3]:
started=time.monotonic(); translations,capped=greedy_translations(model,tok,TEXTS,cfg,batch_size=4)
display(pd.DataFrame({'Source':TEXTS,'English translation':translations,'Generation limit reached':capped}))
print(f'Elapsed: {time.monotonic()-started:.2f} seconds')


,Source,English translation,Generation limit reached
0,Tôi đang học tiếng Anh.,I am learning English.,False
1,私は英語を勉強しています。,I am studying English.,False
2,Cảm ơn bạn rất nhiều.,Thank you very much.,False
3,どうもありがとうございます。,Thank you very much.,False


Elapsed: 0.79 seconds
